In [21]:

import re
import calendar
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import geopandas as gpd

PROJECT_DIR = Path.cwd()

WIKI_INTERIM_DIR = (
    PROJECT_DIR / "data" / "interim" / "wikipedia_earthquake_list"
)

month_numbers = {
    month: number
    for number, month in enumerate(calendar.month_name)
    if month
}

# Match only monthly tables, excluding By_death_toll and By_magnitude
month_pattern = re.compile(
    r"^(January|February|March|April|May|June|"
    r"July|August|September|October|November|December)"
    r"_(\d{4})(?:_part\d+)?$",
    flags=re.IGNORECASE
)

monthly_files = []

for year_dir in sorted(WIKI_INTERIM_DIR.glob("earthquakes_*")):

    folder_match = re.fullmatch(
        r"earthquakes_(\d{4})",
        year_dir.name
    )

    if not folder_match:
        continue

    folder_year = int(folder_match.group(1))

    for file_path in sorted(year_dir.glob("*.csv")):

        match = month_pattern.fullmatch(file_path.stem)

        if not match:
            continue

        month_name = match.group(1).capitalize()
        file_year = int(match.group(2))

        if file_year != folder_year:
            continue

        monthly_files.append({
            "path": file_path,
            "year": file_year,
            "month": month_name,
            "month_number": month_numbers[month_name]
        })

# Create file inventory
file_inventory = []

for item in monthly_files:

    temp = pd.read_csv(
        item["path"],
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    file_inventory.append({
        "year": item["year"],
        "month": item["month"],
        "month_number": item["month_number"],
        "filename": item["path"].name,
        "rows": len(temp),
        "columns": ", ".join(temp.columns.astype(str))
    })

monthly_inventory = pd.DataFrame(file_inventory)

print("Total monthly files:", len(monthly_files))
print("Expected monthly files:", 26 * 12)

print(
    "Years covered:",
    sorted(monthly_inventory["year"].unique())
    if not monthly_inventory.empty else []
)

print("\nMonthly file inventory:")
display(monthly_inventory)

# Check missing months
expected_months = {
    (year, month)
    for year in range(2000, 2026)
    for month in range(1, 13)
}

available_months = {
    (item["year"], item["month_number"])
    for item in monthly_files
}

missing_months = sorted(expected_months - available_months)

print("\nMissing monthly files:", len(missing_months))
display(pd.DataFrame(
    missing_months,
    columns=["year", "month_number"]
))


Total monthly files: 309
Expected monthly files: 312
Years covered: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Monthly file inventory:


,year,month,month_number,filename,rows,columns
0,2000,April,4,April_2000.csv,9,"year, Date, Country and location, Mw, Depth (k..."
1,2000,August,8,August_2000.csv,19,"year, Date, Country and location, Mw, Depth (k..."
2,2000,December,12,December_2000.csv,16,"year, Date, Country and location, Mw, Depth (k..."
3,2000,February,2,February_2000.csv,12,"year, Date, Country and location, Mw, Depth (k..."
4,2000,January,1,January_2000.csv,21,"year, Date, Country and location, Mw, Depth (k..."
...,...,...,...,...,...,...
304,2025,March,3,March_2025.csv,31,"year, Date, Country and location, Mw, Depth (k..."
305,2025,May,5,May_2025.csv,34,"year, Date, Country and location, Mw, Depth (k..."
306,2025,November,11,November_2025.csv,23,"year, Date, Country and location, Mw, Depth (k..."
307,2025,October,10,October_2025.csv,29,"year, Date, Country and location, Mw, Depth (k..."



Missing monthly files: 3


,year,month_number
0,2008,10
1,2008,11
2,2008,12


In [22]:
# Inspect all files available for 2008

year_2008_dir = WIKI_INTERIM_DIR / "earthquakes_2008"

files_2008 = sorted(year_2008_dir.glob("*.csv"))

print("Files available for 2008:")
display(pd.DataFrame({
    "filename": [f.name for f in files_2008],
    "size_kb": [round(f.stat().st_size / 1024, 2) for f in files_2008]
}))

# Inspect the annual death-toll table
death_toll_path = year_2008_dir / "By_death_toll_2008.csv"

if death_toll_path.exists():
    df_death_toll_2008 = pd.read_csv(death_toll_path)

    print("\nDeath-toll table shape:", df_death_toll_2008.shape)
    print("\nColumns:")
    print(df_death_toll_2008.columns.tolist())

    display(df_death_toll_2008.head(20))
else:
    print("\nBy_death_toll_2008.csv not found.")

Files available for 2008:


,filename,size_kb
0,April_2008.csv,0.93
1,August_2008.csv,0.83
2,By_death_toll_2008.csv,0.63
3,By_magnitude_2008.csv,0.87
4,February_2008.csv,0.77
5,January_2008.csv,0.88
6,July_2008.csv,0.48
7,June_2008.csv,0.37
8,March_2008.csv,0.79
9,May_2008.csv,0.45



Death-toll table shape: (9, 8)

Columns:
['year', 'Rank', 'Death toll', 'Magnitude', 'Location', 'MMI', 'Depth (km)', 'Date']


,year,Rank,Death toll,Magnitude,Location,MMI,Depth (km),Date
0,2008,1,"87,587",7.9,"China, Sichuan",XI (Extreme),19.0,May 12
1,2008,2,215,6.4,"Pakistan, Balochistan",VIII (Severe),10.0,October 29
2,2008,3,75,6.6,"Kyrgyzstan, Osh",VII (Very strong),27.6,October 5
3,2008,4,44,5.9,"DR Congo, Lake Kivu",VIII (Severe),10.0,February 3
4,2008,5,41,6.1,"China, Sichuan",VI (Strong),10.0,August 30
5,2008,6,23,6.9,"Japan, Honshu",IX (Violent),8.0,June 13
6,2008,7,13,5.8,"Russia, Dagestan",VII (Very strong),20.0,October 11
7,2008,8,11,5.9,"Colombia, El Calvario",VII (Very strong),35.0,May 24
8,2008,9,10,6.4,"China, Tibet",VII (Very strong),10.0,October 6


In [23]:
monthly_files = []
ranking_files = []

monthly_pattern = re.compile(
    r"^(January|February|March|April|May|June|July|August|"
    r"September|October|November|December)"
    r"_(\d{4})(?:_part\d+)?$",
    flags=re.IGNORECASE
)

ranking_pattern = re.compile(
    r"^By_(death_toll|magnitude)_(\d{4})$",
    flags=re.IGNORECASE
)

for year_dir in sorted(WIKI_INTERIM_DIR.glob("earthquakes_*")):

    if not year_dir.is_dir():
        continue

    folder_year = int(year_dir.name.split("_")[-1])

    for file_path in sorted(year_dir.glob("*.csv")):

        monthly_match = monthly_pattern.fullmatch(file_path.stem)
        ranking_match = ranking_pattern.fullmatch(file_path.stem)

        if monthly_match:
            month_name = monthly_match.group(1).capitalize()
            year = int(monthly_match.group(2))

            if year == folder_year:
                monthly_files.append({
                    "path": file_path,
                    "year": year,
                    "month": month_name,
                    "month_number": month_numbers[month_name]
                })

        elif ranking_match:
            ranking_type = ranking_match.group(1).lower()
            year = int(ranking_match.group(2))

            if year == folder_year:
                ranking_files.append({
                    "path": file_path,
                    "year": year,
                    "type": ranking_type
                })

        # All other CSV files are deliberately ignored.

print("Monthly files:", len(monthly_files))
print("Death-toll files:", sum(
    f["type"] == "death_toll" for f in ranking_files
))
print("Magnitude files:", sum(
    f["type"] == "magnitude" for f in ranking_files
))

print("Excluded files: Number_of_earthquakes_worldwide and others.")

Monthly files: 309
Death-toll files: 26
Magnitude files: 26
Excluded files: Number_of_earthquakes_worldwide and others.


In [24]:
from datetime import datetime
from dateutil import parser as date_parser

def parse_ranking_date(value, year):

    if pd.isna(value) or not str(value).strip():
        return pd.NaT

    date_text = str(value).strip()

    date_text = re.sub(r"\[[^\]]*\]", "", date_text)
    date_text = re.sub(
        r"(\d+)(st|nd|rd|th)\b",
        r"\1",
        date_text,
        flags=re.IGNORECASE
    )

    try:
        parsed = date_parser.parse(
            date_text,
            default=datetime(int(year), 1, 1),
            dayfirst=True,
            fuzzy=False
        )

        # Reject dates whose explicit year conflicts
        # with the year column.
        if parsed.year != int(year):
            return pd.NaT

        return pd.Timestamp(parsed)

    except (ValueError, TypeError, OverflowError):
        return pd.NaT

In [26]:

from dateutil import parser as date_parser
from datetime import datetime

WIKI_SEMICLEAN_DIR = (
    PROJECT_DIR / "data" / "interim" / "semicleaned_wikipedia"
)
WIKI_SEMICLEAN_DIR.mkdir(parents=True, exist_ok=True)

FINAL_COLUMNS = [
    "event_date",
    "year",
    "month",
    "month_number",
    "day",
    "country_location",
    "magnitude_mw",
    "mmi",
    "deaths",
    "injured"
]

MONTH_NAMES = list(calendar.month_name)[1:]


def normalise_column(col):
    return re.sub(r"[^a-z0-9]", "", str(col).lower())


def find_column(df, candidates):
    column_lookup = {
        normalise_column(col): col
        for col in df.columns
    }

    for candidate in candidates:
        key = normalise_column(candidate)

        if key in column_lookup:
            return column_lookup[key]

    return None


def clean_numeric(series):
    values = (
        series.astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace(r"\[[^\]]*\]", "", regex=True)
        .replace({
            "": pd.NA,
            "-": pd.NA,
            "—": pd.NA,
            "–": pd.NA
        })
    )

    return pd.to_numeric(values, errors="coerce")


def clean_mmi(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\[[^\]]*\]", "", regex=True)
        .replace({
            "": pd.NA,
            "-": pd.NA,
            "—": pd.NA,
            "–": pd.NA
        })
    )


def parse_monthly_date(value, year, month_number):

    if pd.isna(value) or not str(value).strip():
        return pd.NaT

    value = str(value).strip()

    if re.fullmatch(r"\d{1,2}(?:\.0+)?", value):

        day = int(float(value))

        try:
            return pd.Timestamp(
                year=int(year),
                month=int(month_number),
                day=day
            )
        except ValueError:
            return pd.NaT

    try:
        parsed = date_parser.parse(
            value,
            default=datetime(int(year), int(month_number), 1),
            dayfirst=True
        )

        if parsed.year != int(year):
            return pd.NaT

        return pd.Timestamp(
            year=int(year),
            month=int(month_number),
            day=parsed.day
        )

    except (ValueError, TypeError, OverflowError):
        return pd.NaT


def parse_ranking_date(value, year):

    if pd.isna(value) or not str(value).strip():
        return pd.NaT

    value = str(value).strip()

    value = re.sub(r"\[[^\]]*\]", "", value)

    value = re.sub(
        r"(\d+)(st|nd|rd|th)\b",
        r"\1",
        value,
        flags=re.IGNORECASE
    )

    try:
        parsed = date_parser.parse(
            value,
            default=datetime(int(year), 1, 1),
            dayfirst=True
        )

        if parsed.year != int(year):
            return pd.NaT

        return pd.Timestamp(parsed.date())

    except (ValueError, TypeError, OverflowError):
        return pd.NaT


def standardise_file(raw, year, file_type, month_number=None):

    date_col = find_column(raw, ["Date"])

    location_col = find_column(
        raw,
        ["Country and location", "Location"]
    )

    magnitude_col = find_column(
        raw,
        ["Mw", "Magnitude"]
    )

    mmi_col = find_column(
        raw,
        ["MMI", "Modified Mercalli Intensity"]
    )

    deaths_col = find_column(
        raw,
        [
            "Casualties Dead",
            "Death toll",
            "Deaths",
            "Dead",
            "Fatalities"
        ]
    )

    injured_col = find_column(
        raw,
        [
            "Casualties Injured",
            "Injured",
            "Injuries",
            "Wounded"
        ]
    )

    generic_casualty_cols = [
        col for col in raw.columns
        if normalise_column(col).startswith("casualties")
    ]

    if deaths_col is None and len(generic_casualty_cols) >= 2:
        deaths_col = generic_casualty_cols[0]

    if injured_col is None and len(generic_casualty_cols) >= 2:
        injured_col = generic_casualty_cols[1]

    if location_col is None:
        return pd.DataFrame(columns=FINAL_COLUMNS)

    locations = (
        raw[location_col]
        .astype("string")
        .str.strip()
    )

    magnitude = (
        clean_numeric(raw[magnitude_col])
        if magnitude_col is not None
        else pd.Series(np.nan, index=raw.index)
    )

    mmi = (
        clean_mmi(raw[mmi_col])
        if mmi_col is not None
        else pd.Series(pd.NA, index=raw.index, dtype="string")
    )

    deaths = (
        clean_numeric(raw[deaths_col])
        if deaths_col is not None
        else pd.Series(np.nan, index=raw.index)
    )

    injured = (
        clean_numeric(raw[injured_col])
        if injured_col is not None
        else pd.Series(np.nan, index=raw.index)
    )

    if date_col is not None and file_type == "monthly":

        event_dates = raw[date_col].apply(
            lambda x: parse_monthly_date(
                x, year, month_number
            )
        )

    elif date_col is not None:

        event_dates = raw[date_col].apply(
            lambda x: parse_ranking_date(x, year)
        )

    else:
        event_dates = pd.Series(pd.NaT, index=raw.index)

    event_dates = pd.to_datetime(event_dates)

    result = pd.DataFrame({
        "event_date": event_dates,
        "year": int(year),
        "month": [
            calendar.month_name[d.month]
            if pd.notna(d)
            else (
                calendar.month_name[month_number]
                if month_number is not None
                else pd.NA
            )
            for d in event_dates
        ],
        "month_number": [
            d.month if pd.notna(d) else month_number
            for d in event_dates
        ],
        "day": event_dates.dt.day,
        "country_location": locations,
        "magnitude_mw": magnitude,
        "mmi": mmi,
        "deaths": deaths,
        "injured": injured
    })

    result = result.loc[
        result["country_location"].notna()
        & result["country_location"].ne("")
    ].copy()

    return result[FINAL_COLUMNS]


# Process each year independently

yearly_summaries = []

for year in range(2000, 2026):

    year_dir = WIKI_INTERIM_DIR / f"earthquakes_{year}"

    if not year_dir.exists():
        print(f"{year}: folder not found")
        continue

    death_files = []
    magnitude_files = []
    monthly_file_map = {}

    for file_path in year_dir.glob("*.csv"):

        stem = file_path.stem

        if stem == f"By_death_toll_{year}":
            death_files.append(file_path)

        elif stem == f"By_magnitude_{year}":
            magnitude_files.append(file_path)

        else:
            month_match = re.fullmatch(
                r"([A-Za-z]+)_(\d{4})(?:_part\d+)?",
                stem,
                flags=re.IGNORECASE
            )

            if month_match:

                month_name = month_match.group(1).capitalize()
                file_year = int(month_match.group(2))

                if file_year == year and month_name in MONTH_NAMES:
                    monthly_file_map.setdefault(
                        MONTH_NAMES.index(month_name) + 1,
                        []
                    ).append(file_path)

    ordered_files = (
        [(f, "death_toll", None) for f in sorted(death_files)]
        + [(f, "magnitude", None) for f in sorted(magnitude_files)]
        + [
            (f, "monthly", month_number)
            for month_number in sorted(monthly_file_map)
            for f in sorted(monthly_file_map[month_number])
        ]
    )

    year_records = []

    death_rows = 0
    magnitude_rows = 0
    monthly_rows = 0

    for file_path, file_type, month_number in ordered_files:

        raw = pd.read_csv(
            file_path,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig"
        )

        raw.columns = raw.columns.astype(str).str.strip()

        cleaned = standardise_file(
            raw,
            year,
            file_type,
            month_number
        )

        year_records.append(cleaned)

        if file_type == "death_toll":
            death_rows += len(cleaned)

        elif file_type == "magnitude":
            magnitude_rows += len(cleaned)

        else:
            monthly_rows += len(cleaned)

    if year_records:
        yearly_df = pd.concat(
            year_records,
            ignore_index=True
        )
    else:
        yearly_df = pd.DataFrame(columns=FINAL_COLUMNS)

    yearly_df = yearly_df.sort_values(
        ["event_date", "country_location"],
        na_position="last",
        kind="stable"
    ).reset_index(drop=True)

    output_path = (
        WIKI_SEMICLEAN_DIR
        / f"wikipedia_earthquakes_{year}_semicleaned.csv"
    )

    yearly_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    yearly_summaries.append({
        "year": year,
        "death_toll_rows": death_rows,
        "magnitude_rows": magnitude_rows,
        "monthly_rows": monthly_rows,
        "total_rows": len(yearly_df),
        "mmi_available": yearly_df["mmi"].notna().sum(),
        "deaths_available": yearly_df["deaths"].notna().sum(),
        "injuries_available": yearly_df["injured"].notna().sum(),
        "output_file": output_path.name
    })

    print(
        f"{year} | "
        f"Death toll: {death_rows} | "
        f"Magnitude: {magnitude_rows} | "
        f"Monthly: {monthly_rows} | "
        f"MMI available: {yearly_df['mmi'].notna().sum()} | "
        f"Total: {len(yearly_df)} | Saved"
    )


yearly_summaries = pd.DataFrame(yearly_summaries)

print("\nYEAR-WISE EXTRACTION SUMMARY")
display(yearly_summaries)

print(
    "\nTotal records extracted:",
    yearly_summaries["total_rows"].sum()
)

print("\nYearly files saved in:")
print(WIKI_SEMICLEAN_DIR)


2000 | Death toll: 4 | Magnitude: 16 | Monthly: 194 | MMI available: 165 | Total: 214 | Saved
2001 | Death toll: 4 | Magnitude: 15 | Monthly: 164 | MMI available: 145 | Total: 183 | Saved
2002 | Death toll: 10 | Magnitude: 13 | Monthly: 174 | MMI available: 148 | Total: 197 | Saved
2003 | Death toll: 8 | Magnitude: 15 | Monthly: 186 | MMI available: 159 | Total: 209 | Saved
2004 | Death toll: 10 | Magnitude: 16 | Monthly: 209 | MMI available: 176 | Total: 235 | Saved
2005 | Death toll: 7 | Magnitude: 11 | Monthly: 189 | MMI available: 162 | Total: 207 | Saved
2006 | Death toll: 4 | Magnitude: 11 | Monthly: 143 | MMI available: 4 | Total: 158 | Saved
2007 | Death toll: 7 | Magnitude: 18 | Monthly: 178 | MMI available: 114 | Total: 203 | Saved
2008 | Death toll: 9 | Magnitude: 12 | Monthly: 112 | MMI available: 21 | Total: 133 | Saved
2009 | Death toll: 7 | Magnitude: 17 | Monthly: 199 | MMI available: 149 | Total: 223 | Saved
2010 | Death toll: 7 | Magnitude: 21 | Monthly: 180 | MMI ava

,year,death_toll_rows,magnitude_rows,monthly_rows,total_rows,mmi_available,deaths_available,injuries_available,output_file
0,2000,4,16,194,214,165,40,38,wikipedia_earthquakes_2000_semicleaned.csv
1,2001,4,15,164,183,145,38,33,wikipedia_earthquakes_2001_semicleaned.csv
2,2002,10,13,174,197,148,57,41,wikipedia_earthquakes_2002_semicleaned.csv
3,2003,8,15,186,209,159,53,38,wikipedia_earthquakes_2003_semicleaned.csv
4,2004,10,16,209,235,176,60,55,wikipedia_earthquakes_2004_semicleaned.csv
5,2005,7,11,189,207,162,43,43,wikipedia_earthquakes_2005_semicleaned.csv
6,2006,4,11,143,158,4,30,28,wikipedia_earthquakes_2006_semicleaned.csv
7,2007,7,18,178,203,114,44,24,wikipedia_earthquakes_2007_semicleaned.csv
8,2008,9,12,112,133,21,43,5,wikipedia_earthquakes_2008_semicleaned.csv
9,2009,7,17,199,223,149,50,41,wikipedia_earthquakes_2009_semicleaned.csv



Total records extracted: 5694

Yearly files saved in:
d:\DA2402\Project\EarthQuake\data\interim\semicleaned_wikipedia
